### Imports and device

In [1]:
import torch
import matplotlib.pyplot as plt

from data_preprocessing import create_dataloader, denormalize
from models import Generator, PatchGANDiscriminator

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 4090


### Dataset paths

In [2]:
from pathlib import Path

DATASET_ROOT = Path(
    "/app/DATA266-Lab1-Pair45/task3_GAN/dataset/"
)

MONET_SOURCE = DATASET_ROOT / "monet_jpg"
PHOTO_SOURCE = DATASET_ROOT / "photo_jpg"
SPLIT_ROOT = DATASET_ROOT / "split_data"

print("Dataset root:", DATASET_ROOT)
print("Monet source:", MONET_SOURCE)
print("Photo source:", PHOTO_SOURCE)
print("Split root:", SPLIT_ROOT)

print("Monet folder exists:", MONET_SOURCE.exists())
print("Photo folder exists:", PHOTO_SOURCE.exists())
print("Split folder exists:", SPLIT_ROOT.exists())

Dataset root: /app/DATA266-Lab1-Pair45/task3_GAN/dataset
Monet source: /app/DATA266-Lab1-Pair45/task3_GAN/dataset/monet_jpg
Photo source: /app/DATA266-Lab1-Pair45/task3_GAN/dataset/photo_jpg
Split root: /app/DATA266-Lab1-Pair45/task3_GAN/dataset/split_data
Monet folder exists: True
Photo folder exists: True
Split folder exists: True


### Split the data

In [3]:
# import random
# import shutil


# def split_images(
#     source_dir,
#     domain_name,
#     output_root,
#     train_ratio=0.8,
#     validation_ratio=0.1,
#     seed=42
# ):
#     image_extensions = {
#         ".jpg",
#         ".jpeg",
#         ".png",
#         ".bmp",
#         ".webp"
#     }

#     image_paths = sorted([
#         path for path in source_dir.rglob("*")
#         if path.is_file()
#         and path.suffix.lower() in image_extensions
#     ])

#     if len(image_paths) == 0:
#         raise ValueError(
#             f"No images found in: {source_dir}"
#         )

#     rng = random.Random(seed)
#     rng.shuffle(image_paths)

#     total_images = len(image_paths)

#     train_end = int(total_images * train_ratio)

#     validation_end = int(
#         total_images
#         * (train_ratio + validation_ratio)
#     )

#     split_paths = {
#         "train": image_paths[:train_end],

#         "validation": image_paths[
#             train_end:validation_end
#         ],

#         "test": image_paths[
#             validation_end:
#         ]
#     }

#     for split_name, paths in split_paths.items():

#         destination_dir = (
#             output_root
#             / split_name
#             / domain_name
#         )

#         destination_dir.mkdir(
#             parents=True,
#             exist_ok=True
#         )

#         for image_path in paths:
#             shutil.copy2(
#                 image_path,
#                 destination_dir / image_path.name
#             )

#         print(
#             f"{domain_name} | "
#             f"{split_name}: {len(paths)} images"
#         )


# # Split photos independently
# split_images(
#     source_dir=PHOTO_SOURCE,
#     domain_name="photos",
#     output_root=SPLIT_ROOT,
#     seed=42
# )

# # Split Monet images independently
# split_images(
#     source_dir=MONET_SOURCE,
#     domain_name="monet",
#     output_root=SPLIT_ROOT,
#     seed=43
# )

photos | train: 5630 images
photos | validation: 704 images
photos | test: 704 images
monet | train: 240 images
monet | validation: 30 images
monet | test: 30 images


### Create the DataLoader

In [9]:
BATCH_SIZE = 10

train_loader = create_dataloader(
    domain_a_dir=SPLIT_ROOT / "train" / "photos",
    domain_b_dir=SPLIT_ROOT / "train" / "monet",
    batch_size=BATCH_SIZE,
    train=True,
    shuffle=True,
    num_workers=0
)

print("Batch size:", BATCH_SIZE)
print("Number of training batches:", len(train_loader))

Batch size: 10
Number of training batches: 563


### Create the four CycleGAN networks

In [4]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

G_AB = Generator(
    input_channels=3,
    output_channels=3,
    num_residual_blocks=9
).to(device)

G_BA = Generator(
    input_channels=3,
    output_channels=3,
    num_residual_blocks=9
).to(device)

D_A = PatchGANDiscriminator(
    input_channels=3
).to(device)

D_B = PatchGANDiscriminator(
    input_channels=3
).to(device)

print("Models created successfully.")
print("Device:", device)

Models created successfully.
Device: cuda


### Define losses

In [1]:
import torch.nn as nn

# Adversarial loss
# It encourages generated images to look realistic to the discriminator.
criterion_gan = nn.MSELoss()

# Cycle-consistency loss
# This checks whether the image can be translated and then returned close to its original form
criterion_cycle = nn.L1Loss()

# Identity loss
# This checks whether a generator preserves an image when it already receives an image from its target domain
criterion_identity = nn.L1Loss()

# Loss weights
lambda_cycle = 10.0
lambda_identity = 5.0

### Optimizers

In [9]:
import itertools
import torch.optim as optim

learning_rate = 0.0002
beta1 = 0.5
beta2 = 0.999

# One optimizer for both generators
optimizer_G = optim.Adam(
    itertools.chain(
        G_AB.parameters(),
        G_BA.parameters()
    ),
    lr=learning_rate,
    betas=(beta1, beta2)
)

# Separate optimizer for each discriminator
optimizer_D_A = optim.Adam(
    D_A.parameters(),
    lr=learning_rate,
    betas=(beta1, beta2)
)

optimizer_D_B = optim.Adam(
    D_B.parameters(),
    lr=learning_rate,
    betas=(beta1, beta2)
)

- optimizer_G   → updates both generators
- optimizer_D_A → updates the photo discriminator
- optimizer_D_B → updates the Monet discriminator

### Testing on one sample

In [10]:
batch = next(iter(train_loader))

real_A = batch["A"].to(device)
real_B = batch["B"].to(device)

print("Photo batch shape:", real_A.shape)
print("Monet batch shape:", real_B.shape)
print("Photo value range:", real_A.min().item(), real_A.max().item())
print("Monet value range:", real_B.min().item(), real_B.max().item())

with torch.no_grad():
    fake_B = G_AB(real_A)
    fake_A = G_BA(real_B)

print("Generated Monet shape:", fake_B.shape)
print("Generated Photo shape:", fake_A.shape)
print("Generated value range:", fake_B.min().item(), fake_B.max().item())

Photo batch shape: torch.Size([10, 3, 256, 256])
Monet batch shape: torch.Size([10, 3, 256, 256])
Photo value range: -1.0 1.0
Monet value range: -1.0 1.0
Generated Monet shape: torch.Size([10, 3, 256, 256])
Generated Photo shape: torch.Size([10, 3, 256, 256])
Generated value range: -0.9999663233757019 0.9999776482582092


### Train the Model

In [11]:
import os
import time
import torch


# ============================================================
# Training settings
# ============================================================

num_epochs = 100
patience = 10

checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

batch_size_used = train_loader.batch_size

print("Checkpoint folder:", os.path.abspath(checkpoint_dir))
print("Batch size:", batch_size_used)


# ============================================================
# Clear unused GPU memory
# ============================================================

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# AMP setup
# ============================================================

use_amp = torch.cuda.is_available()

scaler = torch.cuda.amp.GradScaler(
    enabled=use_amp
)

print("AMP enabled:", use_amp)


# ============================================================
# Helper functions
# ============================================================

def set_requires_grad(models, requires_grad):
    for model in models:
        for parameter in model.parameters():
            parameter.requires_grad = requires_grad


def safe_torch_save(checkpoint, filename):
    """
    Save to a temporary file first, then replace
    the final checkpoint only after saving succeeds.
    """
    temporary_filename = filename + ".tmp"

    torch.save(
        checkpoint,
        temporary_filename
    )

    os.replace(
        temporary_filename,
        filename
    )


# ============================================================
# Loss history
# ============================================================

history = {
    "generator": [],
    "discriminator_A": [],
    "discriminator_B": [],
    "cycle": [],
    "identity": []
}


# ============================================================
# Early stopping variables
# ============================================================

best_monitor_score = float("inf")
epochs_without_improvement = 0


# ============================================================
# Start training
# ============================================================

training_start_time = time.time()

print("Training started.", flush=True)
print(f"Total epochs: {num_epochs}", flush=True)
print(f"Batches per epoch: {len(train_loader)}", flush=True)
print(f"Device: {device}", flush=True)
print("-" * 70, flush=True)


for epoch in range(num_epochs):

    epoch_start_time = time.time()

    total_loss_G = 0.0
    total_loss_D_A = 0.0
    total_loss_D_B = 0.0
    total_loss_cycle = 0.0
    total_loss_identity = 0.0

    number_of_batches = 0

    print(
        f"Starting Epoch [{epoch + 1}/{num_epochs}]...",
        flush=True
    )


    # ========================================================
    # Process every batch
    # ========================================================

    for batch in train_loader:

        real_A = batch["A"].to(device)
        real_B = batch["B"].to(device)


        # ====================================================
        # Step 1: Train both generators
        # ====================================================

        set_requires_grad([D_A, D_B], False)

        optimizer_G.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):

            # Photo → Monet
            fake_B = G_AB(real_A)

            # Monet → Photo
            fake_A = G_BA(real_B)

            # Photo → Monet → Photo
            reconstructed_A = G_BA(fake_B)

            # Monet → Photo → Monet
            reconstructed_B = G_AB(fake_A)

            # Identity mappings
            identity_A = G_BA(real_A)
            identity_B = G_AB(real_B)

            # Discriminator predictions for generated images
            prediction_fake_B = D_B(fake_B)
            prediction_fake_A = D_A(fake_A)

            # Generator adversarial losses
            loss_G_AB = criterion_gan(
                prediction_fake_B,
                torch.ones_like(prediction_fake_B)
            )

            loss_G_BA = criterion_gan(
                prediction_fake_A,
                torch.ones_like(prediction_fake_A)
            )

            # Cycle-consistency losses
            loss_cycle_A = criterion_cycle(
                reconstructed_A,
                real_A
            )

            loss_cycle_B = criterion_cycle(
                reconstructed_B,
                real_B
            )

            loss_cycle_total = (
                loss_cycle_A + loss_cycle_B
            )

            # Identity losses
            loss_identity_A = criterion_identity(
                identity_A,
                real_A
            )

            loss_identity_B = criterion_identity(
                identity_B,
                real_B
            )

            loss_identity_total = (
                loss_identity_A + loss_identity_B
            )

            # Total generator loss
            loss_G = (
                loss_G_AB
                + loss_G_BA
                + lambda_cycle * loss_cycle_total
                + lambda_identity * loss_identity_total
            )

        scaler.scale(loss_G).backward()
        scaler.step(optimizer_G)


        # ====================================================
        # Step 2: Train discriminator D_A
        # ====================================================

        set_requires_grad([D_A, D_B], True)

        optimizer_D_A.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):

            prediction_real_A = D_A(real_A)
            prediction_fake_A = D_A(fake_A.detach())

            loss_D_A_real = criterion_gan(
                prediction_real_A,
                torch.ones_like(prediction_real_A)
            )

            loss_D_A_fake = criterion_gan(
                prediction_fake_A,
                torch.zeros_like(prediction_fake_A)
            )

            loss_D_A = 0.5 * (
                loss_D_A_real + loss_D_A_fake
            )

        scaler.scale(loss_D_A).backward()
        scaler.step(optimizer_D_A)


        # ====================================================
        # Step 3: Train discriminator D_B
        # ====================================================

        optimizer_D_B.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):

            prediction_real_B = D_B(real_B)
            prediction_fake_B = D_B(fake_B.detach())

            loss_D_B_real = criterion_gan(
                prediction_real_B,
                torch.ones_like(prediction_real_B)
            )

            loss_D_B_fake = criterion_gan(
                prediction_fake_B,
                torch.zeros_like(prediction_fake_B)
            )

            loss_D_B = 0.5 * (
                loss_D_B_real + loss_D_B_fake
            )

        scaler.scale(loss_D_B).backward()
        scaler.step(optimizer_D_B)

        # Update AMP scaling once after all optimizers
        scaler.update()


        # ====================================================
        # Store batch losses
        # ====================================================

        total_loss_G += loss_G.item()
        total_loss_D_A += loss_D_A.item()
        total_loss_D_B += loss_D_B.item()
        total_loss_cycle += loss_cycle_total.item()
        total_loss_identity += loss_identity_total.item()

        number_of_batches += 1


        # ====================================================
        # Print batch progress
        # ====================================================

        print(
            f"Epoch [{epoch + 1}/{num_epochs}] | "
            f"Batch [{number_of_batches}/{len(train_loader)}] | "
            f"Generator: {loss_G.item():.4f} | "
            f"D_A: {loss_D_A.item():.4f} | "
            f"D_B: {loss_D_B.item():.4f} | "
            f"Cycle: {loss_cycle_total.item():.4f} | "
            f"Identity: {loss_identity_total.item():.4f}",
            flush=True
        )


        # Release large tensor references
        del (
            real_A,
            real_B,
            fake_A,
            fake_B,
            reconstructed_A,
            reconstructed_B,
            identity_A,
            identity_B,
            prediction_fake_A,
            prediction_fake_B,
            prediction_real_A,
            prediction_real_B
        )


    # ========================================================
    # Calculate average epoch losses
    # ========================================================

    if number_of_batches == 0:
        print("No batches were processed. Training stopped.")
        break

    average_loss_G = (
        total_loss_G / number_of_batches
    )

    average_loss_D_A = (
        total_loss_D_A / number_of_batches
    )

    average_loss_D_B = (
        total_loss_D_B / number_of_batches
    )

    average_loss_cycle = (
        total_loss_cycle / number_of_batches
    )

    average_loss_identity = (
        total_loss_identity / number_of_batches
    )


    # ========================================================
    # Store epoch history
    # ========================================================

    history["generator"].append(average_loss_G)
    history["discriminator_A"].append(average_loss_D_A)
    history["discriminator_B"].append(average_loss_D_B)
    history["cycle"].append(average_loss_cycle)
    history["identity"].append(average_loss_identity)


    # ========================================================
    # Print epoch summary
    # ========================================================

    epoch_time = time.time() - epoch_start_time

    print("\nEpoch summary:", flush=True)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Time: {epoch_time:.2f} seconds",
        flush=True
    )

    print(
        f"Average Generator loss: "
        f"{average_loss_G:.6f}",
        flush=True
    )

    print(
        f"Average Photo discriminator loss: "
        f"{average_loss_D_A:.6f}",
        flush=True
    )

    print(
        f"Average Monet discriminator loss: "
        f"{average_loss_D_B:.6f}",
        flush=True
    )

    print(
        f"Average Cycle-consistency loss: "
        f"{average_loss_cycle:.6f}",
        flush=True
    )

    print(
        f"Average Identity loss: "
        f"{average_loss_identity:.6f}",
        flush=True
    )


    # ========================================================
    # Save latest model
    # ========================================================

    # Model-only checkpoint saves disk space
    latest_checkpoint = {
        "epoch": epoch + 1,
        "G_AB": G_AB.state_dict(),
        "G_BA": G_BA.state_dict(),
        "D_A": D_A.state_dict(),
        "D_B": D_B.state_dict(),
        "history": history
    }

    latest_path = os.path.join(
        checkpoint_dir,
        "latest_cyclegan.pth"
    )

    safe_torch_save(
        latest_checkpoint,
        latest_path
    )


    # ========================================================
    # Early stopping monitor
    # ========================================================

    monitor_score = (
        average_loss_cycle
        + average_loss_identity
    )


    # ========================================================
    # Save best model
    # ========================================================

    if monitor_score < best_monitor_score:

        best_monitor_score = monitor_score
        epochs_without_improvement = 0

        best_checkpoint = {
            "epoch": epoch + 1,
            "G_AB": G_AB.state_dict(),
            "G_BA": G_BA.state_dict(),
            "D_A": D_A.state_dict(),
            "D_B": D_B.state_dict(),
            "history": history,
            "best_monitor_score": best_monitor_score
        }

        best_path = os.path.join(
            checkpoint_dir,
            "best_cyclegan.pth"
        )

        safe_torch_save(
            best_checkpoint,
            best_path
        )

        print(
            f"Best model saved. "
            f"Monitor score: {best_monitor_score:.6f}",
            flush=True
        )

    else:

        epochs_without_improvement += 1

        print(
            f"No improvement for "
            f"{epochs_without_improvement} epoch(s).",
            flush=True
        )

        if epochs_without_improvement >= patience:

            print(
                "Early stopping triggered.",
                flush=True
            )

            break


    print("-" * 70, flush=True)


# ============================================================
# Training completed
# ============================================================

total_training_time = (
    time.time() - training_start_time
)

print(
    f"Training completed in "
    f"{total_training_time / 60:.2f} minutes.",
    flush=True
)

print(
    f"Latest checkpoint saved at: {latest_path}",
    flush=True
)

if "best_path" in locals():
    print(
        f"Best checkpoint saved at: {best_path}",
        flush=True
    )

Checkpoint folder: /app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi/src/checkpoints
Batch size: 10
AMP enabled: True
Training started.
Total epochs: 100
Batches per epoch: 563
Device: cuda
----------------------------------------------------------------------
Starting Epoch [1/100]...
Epoch [1/100] | Batch [1/563] | Generator: 17.2718 | D_A: 0.6171 | D_B: 0.7147 | Cycle: 0.9896 | Identity: 0.9952
Epoch [1/100] | Batch [2/563] | Generator: 18.2444 | D_A: 2.4551 | D_B: 0.7196 | Cycle: 0.9785 | Identity: 0.9836
Epoch [1/100] | Batch [3/563] | Generator: 16.9903 | D_A: 0.6777 | D_B: 1.1324 | Cycle: 1.0490 | Identity: 1.0273
Epoch [1/100] | Batch [4/563] | Generator: 16.0410 | D_A: 0.7722 | D_B: 0.9145 | Cycle: 0.9609 | Identity: 0.9192
Epoch [1/100] | Batch [5/563] | Generator: 14.6213 | D_A: 0.8315 | D_B: 2.4257 | Cycle: 0.8053 | Identity: 0.8081
Epoch [1/100] | Batch [6/563] | Generator: 13.7899 | D_A: 0.6447 | D_B: 0.9809 | Cycle: 0.7829 | Identity: 0.7891
Epoch [1/100] | Batch [7/563] | Ge

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Epoch [46/100] | Batch [298/563] | Generator: 2.3803 | D_A: 0.2027 | D_B: 0.2190 | Cycle: 0.1248 | Identity: 0.1052
Epoch [46/100] | Batch [299/563] | Generator: 2.5665 | D_A: 0.2421 | D_B: 0.2684 | Cycle: 0.1259 | Identity: 0.1072
Epoch [46/100] | Batch [300/563] | Generator: 2.0411 | D_A: 0.2457 | D_B: 0.2845 | Cycle: 0.1147 | Identity: 0.0962
Epoch [46/100] | Batch [301/563] | Generator: 2.7246 | D_A: 0.2191 | D_B: 0.2701 | Cycle: 0.1335 | Identity: 0.1144
Epoch [46/100] | Batch [302/563] | Generator: 2.3973 | D_A: 0.2217 | D_B: 0.2599 | Cycle: 0.1315 | Identity: 0.1049
Epoch [46/100] | Batch [303/563] | Generator: 2.5821 | D_A: 0.2006 | D_B: 0.2426 | Cycle: 0.1236 | Identity: 0.1099
Epoch [46/100] | Batch [304/563] | Generator: 2.1477 | D_A: 0.2156 | D_B: 0.2755 | Cycle: 0.1194 | Identity: 0.0971
Epoch [46/100] | Batch [305/563] | Generator: 2.7989 | D_A: 0.1961 | D_B: 0.2593 | Cycle: 0.1383 | Identity: 0.1202
Epoch [46/100] | Batch [306/563] | Generator: 2.5372 | D_A: 0.1978 | D_B

### Generate translated test images using the best checkpoint

In [13]:
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch

# --------------------------------------------------
# Dataset containing only one domain
# --------------------------------------------------

class SingleDomainDataset(Dataset):

    def __init__(self, image_dir):
        self.image_paths = sorted([
            path for path in Path(image_dir).rglob("*")
            if path.suffix.lower() in {
                ".jpg", ".jpeg", ".png", ".bmp", ".webp"
            }
        ])

        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize(
                (0.5, 0.5, 0.5),
                (0.5, 0.5, 0.5)
            )
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image = Image.open(
            self.image_paths[index]
        ).convert("RGB")

        return self.transform(image)


# --------------------------------------------------
# Independent test loaders
# --------------------------------------------------

photo_test_dataset = SingleDomainDataset(
    SPLIT_ROOT / "test" / "photos"
)

monet_test_dataset = SingleDomainDataset(
    SPLIT_ROOT / "test" / "monet"
)

photo_test_loader = DataLoader(
    photo_test_dataset,
    batch_size=10,
    shuffle=False,
    num_workers=0
)

monet_test_loader = DataLoader(
    monet_test_dataset,
    batch_size=10,
    shuffle=False,
    num_workers=0
)

print("Test photos:", len(photo_test_dataset))
print("Test Monet paintings:", len(monet_test_dataset))


# --------------------------------------------------
# New output folders
# --------------------------------------------------

OUTPUT_ROOT_V2 = Path("generated_outputs_v2")

PHOTO_TO_MONET_V2 = OUTPUT_ROOT_V2 / "photo_to_monet"
MONET_TO_PHOTO_V2 = OUTPUT_ROOT_V2 / "monet_to_photo"

PHOTO_TO_MONET_V2.mkdir(parents=True, exist_ok=True)
MONET_TO_PHOTO_V2.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------
# Save image helper
# --------------------------------------------------

def save_image_tensor(image_tensor, output_path):

    image_tensor = denormalize(
        image_tensor.detach().cpu()
    )

    image_array = (
        image_tensor.permute(1, 2, 0).numpy() * 255
    ).astype("uint8")

    Image.fromarray(image_array).save(
        output_path,
        format="JPEG",
        quality=95
    )


# --------------------------------------------------
# Generate Photo → Monet
# --------------------------------------------------

G_AB.eval()
G_BA.eval()

with torch.no_grad():

    counter = 0

    for photos in photo_test_loader:

        photos = photos.to(device)
        generated_monets = G_AB(photos)

        for image in generated_monets:

            save_image_tensor(
                image,
                PHOTO_TO_MONET_V2
                / f"photo_to_monet_{counter:05d}.jpg"
            )

            counter += 1

print("Photo → Monet images:", counter)


# --------------------------------------------------
# Generate Monet → Photo
# --------------------------------------------------

with torch.no_grad():

    counter = 0

    for monets in monet_test_loader:

        monets = monets.to(device)
        generated_photos = G_BA(monets)

        for image in generated_photos:

            save_image_tensor(
                image,
                MONET_TO_PHOTO_V2
                / f"monet_to_photo_{counter:05d}.jpg"
            )

            counter += 1

print("Monet → Photo images:", counter)

Test photos: 704
Test Monet paintings: 30
Photo → Monet images: 704
Monet → Photo images: 30


### Calculate the distribution metrics: FID and KID in both directions.

In [17]:
import torch
from pathlib import Path
from PIL import Image
from torchvision import transforms
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

# --------------------------------------------------
# Image loading
# --------------------------------------------------

image_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.PILToTensor()
])


def get_image_paths(folder):
    return sorted([
        path for path in Path(folder).rglob("*")
        if path.suffix.lower() in {
            ".jpg", ".jpeg", ".png", ".bmp", ".webp"
        }
    ])


def load_images(image_paths, batch_size=8):
    batches = []

    for start in range(0, len(image_paths), batch_size):

        batch_paths = image_paths[
            start:start + batch_size
        ]

        images = []

        for path in batch_paths:
            image = Image.open(path).convert("RGB")
            image = image_transform(image)
            images.append(image)

        batches.append(torch.stack(images))

    return batches


# --------------------------------------------------
# Metric calculation
# --------------------------------------------------

def calculate_fid_kid(real_dir, generated_dir):

    real_paths = get_image_paths(real_dir)
    generated_paths = get_image_paths(generated_dir)

    print("Real images:", len(real_paths))
    print("Generated images:", len(generated_paths))

    fid = FrechetInceptionDistance(
        feature=2048,
        normalize=False
    ).to(device)

    subset_size = min(
        50,
        len(real_paths),
        len(generated_paths)
    )

    kid = KernelInceptionDistance(
        feature=2048,
        subset_size=subset_size,
        normalize=False
    ).to(device)

    real_batches = load_images(real_paths)
    generated_batches = load_images(generated_paths)

    with torch.no_grad():

        for batch in real_batches:
            batch = batch.to(device)
            fid.update(batch, real=True)
            kid.update(batch, real=True)

        for batch in generated_batches:
            batch = batch.to(device)
            fid.update(batch, real=False)
            kid.update(batch, real=False)

    fid_score = fid.compute().item()

    kid_mean, kid_std = kid.compute()

    return {
        "FID": fid_score,
        "KID_mean": kid_mean.item(),
        "KID_std": kid_std.item()
    }


# --------------------------------------------------
# Photo → Monet
# --------------------------------------------------

photo_to_monet_metrics = calculate_fid_kid(
    real_dir=SPLIT_ROOT / "test" / "monet",
    generated_dir=PHOTO_TO_MONET_V2
)

print("\nPhoto → Monet metrics")
print(photo_to_monet_metrics)


# --------------------------------------------------
# Monet → Photo
# --------------------------------------------------

monet_to_photo_metrics = calculate_fid_kid(
    real_dir=SPLIT_ROOT / "test" / "photos",
    generated_dir=MONET_TO_PHOTO_V2
)

print("\nMonet → Photo metrics")
print(monet_to_photo_metrics)

Real images: 30
Generated images: 704


Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:06<00:00, 13.8MB/s]
/opt/conda/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `Kernel Inception Distance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)



Photo → Monet metrics
{'FID': 208.01608276367188, 'KID_mean': 0.018572933971881866, 'KID_std': 0.003457213519141078}
Real images: 704
Generated images: 30

Monet → Photo metrics
{'FID': 188.5878448486328, 'KID_mean': 0.024574726819992065, 'KID_std': 0.004919223953038454}


### Getting the results from the trained model

In [20]:
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


class FolderDataset(Dataset):

    def __init__(self, folder):

        self.paths = sorted([
            path for path in Path(folder).rglob("*")
            if path.suffix.lower() in {
                ".jpg", ".jpeg", ".png", ".bmp", ".webp"
            }
        ])

        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize(
                (0.5, 0.5, 0.5),
                (0.5, 0.5, 0.5)
            )
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):

        image = Image.open(
            self.paths[index]
        ).convert("RGB")

        return self.transform(image)


def save_tensor_image(image, output_path):

    image = image.detach().cpu()
    image = image * 0.5 + 0.5
    image = image.clamp(0, 1)

    image_array = (
        image.permute(1, 2, 0).numpy() * 255
    ).astype("uint8")

    Image.fromarray(image_array).save(
        output_path,
        format="JPEG",
        quality=95
    )

print("FolderDataset is ready.")

FolderDataset is ready.


### Results generated from Monets to Photos

In [21]:
from pathlib import Path
from torch.utils.data import DataLoader

# Load the best saved checkpoint
best_checkpoint_path = Path(checkpoint_dir) / "best_cyclegan.pth"

checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device
)

G_AB.load_state_dict(checkpoint["G_AB"])
G_BA.load_state_dict(checkpoint["G_BA"])
D_A.load_state_dict(checkpoint["D_A"])
D_B.load_state_dict(checkpoint["D_B"])

print("Loaded best checkpoint from epoch:", checkpoint["epoch"])

# Read all 300 Monet images
monet_dataset = FolderDataset(MONET_SOURCE)

monet_loader = DataLoader(
    monet_dataset,
    batch_size=10,
    shuffle=False,
    num_workers=0
)

# Save Monet → Photo predictions
MONET_TO_PHOTO = Path("kaggle_outputs/pred_A2B")
MONET_TO_PHOTO.mkdir(parents=True, exist_ok=True)

G_BA.eval()

counter = 0

with torch.no_grad():
    for monets in monet_loader:
        monets = monets.to(device)

        generated_photos = G_BA(monets)

        for image in generated_photos:
            save_tensor_image(
                image,
                MONET_TO_PHOTO / f"monet_to_photo_{counter:05d}.jpg"
            )
            counter += 1

print("Monet → Photo images generated:", counter)

Loaded best checkpoint from epoch: 53
Monet → Photo images generated: 300


### Results generated from Photos to Monet

In [23]:
PHOTO_TO_MONET = Path("kaggle_outputs/pred_B2A")
PHOTO_TO_MONET.mkdir(parents=True, exist_ok=True)

photo_dataset = FolderDataset(PHOTO_SOURCE)

photo_loader = DataLoader(
    photo_dataset,
    batch_size=10,
    shuffle=False,
    num_workers=0
)

G_AB.eval()

counter = 0

with torch.no_grad():
    for photos in photo_loader:
        photos = photos.to(device)

        generated_monets = G_AB(photos)

        for image in generated_monets:
            save_tensor_image(
                image,
                PHOTO_TO_MONET / f"photo_to_monet_{counter:05d}.jpg"
            )

            counter += 1

            if counter == 300:
                break

        if counter == 300:
            break

print("Photo → Monet images generated:", counter)
print("Saved in:", PHOTO_TO_MONET.resolve())

Photo → Monet images generated: 300
Saved in: /app/DATA266-Lab1-Pair45/task3_GAN/Samruddhi/src/kaggle_outputs/pred_B2A
